# 00 · Comprobación del entorno

Ejecutad las celdas **en orden** (Mayús + Intro). Si todas terminan sin error, el entorno está listo.
Si algo falla, consultad la tabla de problemas frecuentes al final o el `README.md`.

**Antes de empezar** (en PowerShell, en la carpeta del repositorio):
```
.\bda.ps1 iniciar
.\bda.ps1 datos 0.1      # datos pequeños para probar (luego: .\bda.ps1 datos)
.\bda.ps1 ingesta
```

In [ ]:
import os, sys, platform
print("Python:", platform.python_version())
from puerto.sesion import crear_spark, executors_vivos
spark = crear_spark("00-comprobacion")
print("Spark:", spark.version, "| master:", spark.sparkContext.master)
print("Executors registrados:", executors_vivos(spark))

Python: 3.10.12


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 14:36:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.8 | master: spark://spark-master:7077
Executors registrados: 0


## 1. ¿Calculan los workers?
Esta celda reparte un cálculo en 8 particiones. Si falla con *«Python in worker has different version»*,
la imagen de Spark no es la misma en todos los contenedores: reconstruidla (`.\bda.ps1 construir`).

In [2]:
rdd = spark.sparkContext.parallelize(range(1_000_000), 8)
print("Suma de cuadrados:", rdd.map(lambda x: x * x).sum())
print("Particiones:", rdd.getNumPartitions())

Suma de cuadrados: 333332833333500000
Particiones: 8


## 2. ¿Se puede escribir y leer en HDFS?

In [3]:
df = spark.range(100_000).withColumnRenamed("id", "n")
df.write.mode("overwrite").parquet("hdfs://namenode:8020/tmp/prueba_bda")
print("Filas leídas de HDFS:", spark.read.parquet("/tmp/prueba_bda").count())

Filas leídas de HDFS: 100000


## 3. ¿Responde WebHDFS y cuántos DataNodes hay?

In [4]:
from puerto import hdfs_web as h
h.ls("/")
for d in h.datanodes():
    print(d)

tipo      réplicas  bloque       tamaño  nombre
DIRECTORY        0    0 MB        0.0 B  puerto
DIRECTORY        0    0 MB        0.0 B  tmp
{'datanode': '77dcae3395d6:9866', 'estado': 'vivo', 'usado': '243.2 MB', 'bloques': 143, 'ultimo_contacto_s': 2}
{'datanode': 'c269c80bf66e:9866', 'estado': 'vivo', 'usado': '243.2 MB', 'bloques': 143, 'ultimo_contacto_s': 2}
{'datanode': '9f08c6a2e4f8:9866', 'estado': 'vivo', 'usado': '243.2 MB', 'bloques': 143, 'ultimo_contacto_s': 2}


## 4. ¿Están los datos generados e ingestados?

In [5]:
print("Datos locales en /datos/raw:", sorted(os.listdir("/datos/raw")) if os.path.isdir("/datos/raw") else "NO HAY (.\\bda.ps1 datos)")
try:
    h.ls("/puerto/raw")
    print(h.resumen("/puerto/raw"))
except Exception as e:
    print("Aún no hay ingesta en HDFS (.\\bda.ps1 ingesta):", e)

Datos locales en /datos/raw: ['MANIFIESTO.sha256', 'ais', 'contenedores', 'escalas', 'maestros', 'negocio', 'sensores']
tipo      réplicas  bloque       tamaño  nombre
DIRECTORY        0    0 MB        0.0 B  ais
DIRECTORY        0    0 MB        0.0 B  contenedores
DIRECTORY        0    0 MB        0.0 B  escalas
DIRECTORY        0    0 MB        0.0 B  maestros
DIRECTORY        0    0 MB        0.0 B  negocio
DIRECTORY        0    0 MB        0.0 B  sensores
{'ficheros': 136, 'carpetas': 7, 'bytes': 251900520, 'bytes_en_disco': 755701560, 'tamano': '240.2 MB', 'en_disco': '720.7 MB'}


In [6]:
spark.stop()

## Problemas frecuentes

| Síntoma | Causa probable | Solución |
|---|---|---|
| `Connection refused namenode:8020` | El NameNode aún arranca | Esperad 30 s y repetid |
| `Initial job has not accepted any resources` | No hay workers o están ocupados por otra sesión | `.\bda.ps1 estado`; cerrad otros notebooks con `spark.stop()` |
| `Python in worker has different version` | Imágenes distintas | `.\bda.ps1 construir` y `docker compose up -d --force-recreate` |
| `/bin/bash^M: bad interpreter` | Scripts con finales de línea de Windows | Volved a clonar (el `.gitattributes` lo evita) |
| El PC va muy lento | Poca memoria para Docker | Revisad `.wslconfig` (guía de instalación) |